# Project 12 GraphGPS Official OGB Benchmark

This private, approval-gated kernel runs the frozen `ogbg-molhiv` scaffold benchmark on Kaggle CUDA. It writes raw per-seed records before aggregation and keeps W&B disabled.

In [ ]:
import base64
import hashlib
import io
import zipfile
from pathlib import Path, PurePosixPath
import sys

EMBEDDED_SOURCE_B64 = '''...'''
SOURCE_ROOT = Path('/kaggle/working/graphgps_upgrade_source')
archive_bytes = base64.b64decode(EMBEDDED_SOURCE_B64)
source_revision = hashlib.sha256(archive_bytes).hexdigest()
if not (SOURCE_ROOT / 'graphgps_bench').is_dir():
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        for info in archive.infolist():
            normalized = PurePosixPath(info.filename.replace('\\', '/'))
            if normalized.is_absolute() or '..' in normalized.parts:
                raise RuntimeError(f'unsafe archive path: {info.filename}')
            target = SOURCE_ROOT.joinpath(*normalized.parts)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(info))
sys.path.insert(0, str(SOURCE_ROOT))
print({'source_root': str(SOURCE_ROOT), 'source_revision': source_revision})


In [ ]:
# APPROVAL REQUIRED BEFORE DEPENDENCY INSTALLATION OR VERIFICATION.
APPROVAL_GRANTED = True
if not APPROVAL_GRANTED:
    raise RuntimeError('Approval required before dependency installation or verification')


In [ ]:
import importlib.util
import os
import re
import subprocess
import sys

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
def run(command):
    return subprocess.run(command, cwd=SOURCE_ROOT, text=True, capture_output=True)

dependency_result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-input', '-r', 'requirements-kaggle.txt'], cwd=SOURCE_ROOT, text=True, capture_output=True)
if dependency_result.returncode != 0:
    raise RuntimeError({'dependency_exit_code': dependency_result.returncode, 'stderr_tail': dependency_result.stderr[-3000:]})
compile_result = run([sys.executable, '-m', 'compileall', '-q', 'graphgps_bench', 'tests'])
test_result = run([sys.executable, '-m', 'pytest', '-p', 'no:cacheprovider', '-q', 'tests'])
test_text = test_result.stdout + test_result.stderr
match = re.search(r'(\d+) passed', test_text)
test_count = int(match.group(1)) if match else 0
if compile_result.returncode != 0 or test_result.returncode != 0:
    raise RuntimeError({'compile_exit_code': compile_result.returncode, 'test_exit_code': test_result.returncode, 'test_output_tail': test_text[-6000:]})
print({'compile_exit_code': compile_result.returncode, 'test_exit_code': test_result.returncode, 'test_count': test_count})


In [ ]:
# APPROVAL REQUIRED BEFORE OGB DOWNLOAD/ACCESS.
if not APPROVAL_GRANTED:
    raise RuntimeError('Approval required before OGB download/access')
from graphgps_bench.config import load_config
config = load_config(SOURCE_ROOT / 'configs' / 'kaggle_benchmark.toml')
config_hash = config.stable_hash()
import torch
import torch_geometric
import ogb
print({'torch': torch.__version__, 'torch_geometric': torch_geometric.__version__, 'ogb': ogb.__version__, 'cuda': torch.cuda.is_available(), 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'config_hash': config_hash})


In [ ]:
# APPROVAL REQUIRED BEFORE GPU EXECUTION/TRAINING.
if not APPROVAL_GRANTED:
    raise RuntimeError('Approval required before GPU execution/training')
from graphgps_bench.benchmark import run_official_benchmark
import os
os.chdir(SOURCE_ROOT)
benchmark_result = run_official_benchmark(config, dataset_root='/kaggle/working/ogb_data')
print(benchmark_result['manifest'])


In [ ]:
import json
from datetime import datetime, timezone
evidence = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'source_revision': source_revision,
    'configuration_hash': config_hash,
    'dependency_versions': benchmark_result['manifest']['versions'],
    'device': benchmark_result['manifest']['device'],
    'hardware': benchmark_result['manifest']['hardware'],
    'seed_set': list(config.seeds),
    'models': list(config.models),
    'split': config.split,
    'test_count': test_count,
    'benchmark_executed': True,
    'gpu_used': True,
    'aggregate': benchmark_result['aggregate'],
    'artifact_paths': benchmark_result['manifest'],
    'restricted_artifact_count': 0,
}
evidence_path = Path('/kaggle/working/graphgps_upgrade_official_evidence.json')
evidence_path.write_text(json.dumps(evidence, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(json.loads(evidence_path.read_text(encoding='utf-8')), indent=2, sort_keys=True))
